In [61]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv


In [62]:
df= pd.read_csv("/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv")

In [63]:
df.head()
df['State'].value_counts()

State
CA    1741433
FL     880192
TX     582837
SC     382557
NY     347960
NC     338199
VA     303301
PA     296620
MN     192084
OR     179660
AZ     170609
GA     169234
IL     168958
TN     167388
MI     162191
LA     149701
NJ     140719
MD     140417
OH     118115
WA     108221
AL     101044
UT      97079
CO      90885
OK      83647
MO      77323
CT      71005
IN      67224
MA      61996
WI      34688
KY      32254
NE      28870
MT      28496
IA      26307
AR      22780
NV      21665
KS      20992
DC      18630
RI      16971
MS      15181
DE      14097
WV      13793
ID      11376
NM      10325
NH      10213
WY       3757
ND       3487
ME       2698
VT        926
SD        289
Name: count, dtype: int64

In [64]:
df_ny= df[df['State'] == 'NY']

In [65]:
df_ny.shape

(347960, 46)

In [66]:
columns= ['Severity', 'Start_Time', 'Start_Lat','Start_Lng',  'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)','Precipitation(in)', 'Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']

In [67]:

df_ny=df_ny[columns]

In [68]:
df_ny.columns

Index(['Severity', 'Start_Time', 'Start_Lat', 'Start_Lng', 'Temperature(F)',
       'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Crossing',
       'Junction', 'Railway', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset'],
      dtype='object')

In [69]:
df_ny.notnull().sum()

Severity             347960
Start_Time           347960
Start_Lat            347960
Start_Lng            347960
Temperature(F)       345892
Wind_Chill(F)        276306
Humidity(%)          345769
Pressure(in)         344976
Visibility(mi)       344879
Wind_Speed(mph)      330081
Precipitation(in)    261449
Weather_Condition    345234
Crossing             347960
Junction             347960
Railway              347960
Stop                 347960
Traffic_Signal       347960
Sunrise_Sunset       347250
dtype: int64

In [70]:
df_ny.drop(columns=['Wind_Chill(F)', 'Precipitation(in)'], inplace=True)

In [71]:
df_ny.fillna({
    'Temperature(F)': df_ny['Temperature(F)'].mean(),
    'Humidity(%)': df_ny['Humidity(%)'].mean(),
    'Pressure(in)': df_ny['Pressure(in)'].mean(),
    'Visibility(mi)': df_ny['Visibility(mi)'].mean(),
    'Wind_Speed(mph)': df_ny['Wind_Speed(mph)'].mean()
},inplace=True)

In [72]:
df_ny.fillna({
    'Weather_Condition': df_ny['Weather_Condition'].mode()[0],
    'Sunrise_Sunset': df_ny['Sunrise_Sunset'].mode()[0]
},inplace=True)

In [73]:
df_ny['Start_Time'] = pd.to_datetime(df_ny['Start_Time'], errors='coerce')
df_ny['hour']= df_ny['Start_Time'].dt.hour
df_ny['day']= df_ny['Start_Time'].dt.dayofweek
df_ny['month']= df_ny['Start_Time'].dt.month

In [74]:
df_ny.sample(12)
df_ny.notnull().sum()
df_ny.fillna({
    'hour': df_ny['hour'].mode()[0],
    'day': df_ny['day'].mode()[0],
    'month': df_ny['month'].mode()[0]
}, inplace=True)

In [75]:
df_ny.notnull().sum()
df_ny.drop(columns= ['Start_Time'], inplace=True)

In [76]:

df_ny.notnull().sum()

Severity             347960
Start_Lat            347960
Start_Lng            347960
Temperature(F)       347960
Humidity(%)          347960
Pressure(in)         347960
Visibility(mi)       347960
Wind_Speed(mph)      347960
Weather_Condition    347960
Crossing             347960
Junction             347960
Railway              347960
Stop                 347960
Traffic_Signal       347960
Sunrise_Sunset       347960
hour                 347960
day                  347960
month                347960
dtype: int64

In [77]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

ohe= OneHotEncoder(handle_unknown='ignore',sparse_output=False)

In [78]:
cat_cols= ['Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']
preprocessor= ColumnTransformer(transformers=[
    ('cat', ohe, cat_cols)
], remainder= 'passthrough')   
 

In [79]:
x= df_ny.drop(columns=['Severity'])
y= df_ny['Severity']


In [80]:
df_ny['Severity'].value_counts()

Severity
2    265902
3     70062
4     10749
1      1247
Name: count, dtype: int64

In [81]:
y1=df_ny['Severity']-1

In [82]:
y1.value_counts()

Severity
1    265902
2     70062
3     10749
0      1247
Name: count, dtype: int64

In [83]:
y1 = (df_ny['Severity'] >= 3).astype(int)

In [84]:
y1.value_counts()

Severity
0    267149
1     80811
Name: count, dtype: int64

In [85]:
import numpy as np

df_ny = df_ny.copy()

# remove impossible coordinates
df_ny = df_ny[
    (df_ny['Start_Lat'].between(-90, 90)) &
    (df_ny['Start_Lng'].between(-180, 180))
]

# remove impossible weather values
df_ny = df_ny[df_ny['Temperature(F)'] > -100]   # sanity check
df_ny = df_ny[df_ny['Wind_Speed(mph)'] >= 0]
df_ny = df_ny[df_ny['Pressure(in)'] > 0]
df_ny = df_ny[df_ny['Humidity(%)'].between(0, 100)]
df_ny = df_ny[df_ny['Visibility(mi)'] >= 0]

In [86]:
def iqr_filter(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return df[(df[col] >= lower) & (df[col] <= upper)]

numeric_cols = [
    'Temperature(F)',
    'Wind_Speed(mph)',
    'Pressure(in)',
    'Visibility(mi)',
    'Humidity(%)'
]

for col in numeric_cols:
    df_ny = iqr_filter(df_ny, col)

In [87]:
lat_min, lat_max = df_ny['Start_Lat'].quantile([0.01, 0.99])
lng_min, lng_max = df_ny['Start_Lng'].quantile([0.01, 0.99])

df_ny = df_ny[
    (df_ny['Start_Lat'].between(lat_min, lat_max)) &
    (df_ny['Start_Lng'].between(lng_min, lng_max))
]

In [88]:
df_ny.shape

(249185, 18)

In [89]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

x_train, x_test, y_train, y_test = train_test_split(
    x, y1, test_size=0.2, random_state=42
)


# pipeline
model2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        objective='binary:logistic',
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=3,
        gamma=0.1,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
    ))
])

# train
model2.fit(x_train, y_train)

# predict (0 or 1)
# y_pred = model2.predict(x_test)
y_prob = model2.predict_proba(x_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

# evaluate
print("Accuracy:", model2.score(x_test, y_test))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.7917001954247614

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.79      0.85     53335
           1       0.54      0.79      0.64     16257

    accuracy                           0.79     69592
   macro avg       0.73      0.79      0.75     69592
weighted avg       0.83      0.79      0.80     69592


Confusion Matrix:
 [[42254 11081]
 [ 3415 12842]]


In [91]:
import joblib
joblib.dump(model2,"ny_model.pkl")

['ny_model.pkl']